<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook Subpixel Vegetation Line Demonstration

The objective of this notebook is to prepare an annual NDVI value and extract a vegetation line from it at different thresholds.

## Step 1.1: Configure the environment

In [1]:
import os
from datetime import datetime
from shapely.geometry import Polygon
from shapely import box
from pyproj import CRS 
import folium
from odc.geo import Geometry
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import xarray as xr
import rioxarray
from ipyleaflet import basemaps
from numpy.lib.stride_tricks import sliding_window_view
import pystac_client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import planetary_computer
# from odc.stac import load
from pystac.client import Client
from skimage.feature import graycomatrix, graycoprops
from utils import load_data, load_s1_dem, scale, calculate_band_indices, apply_mask, mask_land, all_masks, do_prediction

from dea_tools.spatial import subpixel_contours

In [2]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

<font color='red'>1.1. Revise the site of training data...


In [4]:
site = "Mejrirok"
filename = f"inputs/{site}-site.geojson"

## Step 1.2: Configure STAC access and search parameters

Do you remember what a stac catalog is...? You can insert this one : https://stac.digitalearthpacific.org


In [9]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

<font color='red'>1.2. Revise the site name

In [7]:
area_of_interest = "..."

# 1. Read the file
aoi = gpd.read_file(f"{filename}")

# 2. Get the bounding box directly from the GeoDataFrame
# This returns exactly what STAC needs: (minx, miny, maxx, maxy)
bbox = aoi.total_bounds

<font color='red'>1.3. Revise the year to `2025` and include the stac catalog collections `dep_s2_geomad`

In [19]:
# 3. Proceed with your search
year = '2025'
items = list(
    client.search(
        collections=["dep_s2_geomad"],
        bbox=bbox,
        datetime="2017/2025"
    ).items()
)

#### How many images were found?

In [20]:
print(f"Found {len(items)} items in for {datetime}")

Found 9 items in for <class 'datetime.datetime'>


In [21]:
measurements = ["nir", "red"]

In [22]:
data = load_data(
    items,
    measurements,
    bbox,
)

In [23]:
data['nir'] = scale(data.nir)
data['red'] = scale(data.red)
data["ndvi"] = (data["nir"] - data["red"]) / (data["nir"] + data["red"])

Try to adjust the value and rerun each time to see how this affects your vegetation mask. Try to find the best value for identifying vegetation. 

In [25]:
ndvi_threshold = 0.3
ndvi_mask = (data.ndvi.squeeze() > ndvi_threshold).astype('uint8')
ndvi_mask.sel(time='2025').odc.explore(vmin=0, vmax=1)

## Extract vegetation contour lines using the threshold above

and save

In [ ]:
lines = subpixel_contours(data.ndvi, ndvi_threshold)
# Add year as variable
lines['year'] = lines.time.str[0:4]
lines.to_file('vegetation_line.gpkg')

Loading data into memory using Dask
Operating in single z-value, multiple arrays mode


0    2017
1    2018
2    2019
3    2020
4    2021
5    2022
6    2023
7    2024
8    2025
Name: time, dtype: str

### Explore EVI

Add some markdown to explain EVI

Input formula to define the index:   
`data["evi"] = (2.5 * data["nir"] - data["red"]) / (data["nir"] + (6 * data["red"]) - (7.5 * data["blue"]) + 1)`

Repeat the steps taken for NDVI for EVI

### Explore SAVI

Add some markdown to explain SAVI

Input formula to define the index:  
`data["savi"] = (data["nir"] - data["red"]) / (data["nir"] + data["red"])`

Repeat the steps taken for NDVI for SAVI

<font color='red'>1.9. Mask out the 'scaled' variable including the ocean 
>`scaled`, mask = all_masks(`scaled`, return_mask = True)


<font color='red'>Explore the extent of this 'mask' variable  
>`insertvariable`.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

<font color='red'>2.5. Export your final `raster` variable to a geojson file.
>`variable`.to_file(f"output/{site}-vector.geo", index=False)

<font color='red'>2.5. Export your final `vector` variable to a geojson file.
>`variable`.to_file(f"output/{site}-vector.geo", index=False)